# GeoSlide — Phase 6: Model Evaluation & Comparison

This notebook evaluates and compares the classification models trained in the
previous phase of the **GeoSlide** landslide-susceptibility prediction project:

- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes
- Support Vector Machine (SVM)
- Random Forest
- Artificial Neural Network (ANN)

**Scope of this notebook:** evaluation only. Trained model artifacts are
loaded if present. A model is only trained here as a **fallback** if its
saved artifact cannot be found or loaded — evaluation is not blocked on
missing artifacts, but no model is retrained unnecessarily.

**Data:** this notebook reads the raw dataset `wsn_landslide_data.csv`
directly (upload it into the Colab session's file browser before running),
and recreates the exact preprocessing used in Phase 3.

**What this notebook produces:**
1. Accuracy, Precision, Recall, F1-score, and ROC-AUC for every model
2. A confusion matrix for every model
3. A combined ROC curve comparing all models
4. A performance comparison table
5. A data-driven **Best Model Recommendation**
6. A final **Evaluation Summary** covering deployment readiness


## 1. Environment Setup

Import the libraries needed for loading models, computing metrics, and
building visualizations. The classical-ML and Keras imports are also used to
build a fallback model in the rare case a saved artifact is missing.


In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Only used to build a fallback model if a saved artifact is missing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from IPython.display import display, Markdown

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


## 2. Load the Raw Dataset

The dataset is read directly from `wsn_landslide_data.csv`, which should be
uploaded into this Colab session (Files pane → Upload, or drag-and-drop)
before running this cell.


In [ ]:
DATA_PATH = "wsn_landslide_data.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"'{DATA_PATH}' was not found in the Colab session. "
        f"Please upload it (Files pane -> Upload) and re-run this cell."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 3. Recreate Preprocessing (Phase 3)

The preprocessing steps below exactly mirror Phase 3 so the evaluation data
is prepared identically to how the models were trained:

1. Split features (`X`) and target (`y`), where the target column is `Label`.
2. `train_test_split` with `test_size=0.2`, `random_state=42`, stratified by `y`.
3. Fit `StandardScaler` **only on `X_train`**.
4. Transform both `X_train` and `X_test` with that fitted scaler.


In [ ]:
TARGET_COL = "Label"

X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit only on X_train
X_test = scaler.transform(X_test)         # transform X_test with the same scaler

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Class balance in test set:\n{y_test.value_counts(normalize=True).round(4)}")


## 4. Load Trained Models

Each model is loaded from its saved artifact if one is available in the
Colab session (either in a `models/` folder or the current working
directory). **A model is only trained here if its saved artifact cannot be
found or loaded** — this is a fallback, not the default path, and is clearly
flagged in the output below.


In [ ]:
MODEL_FILENAMES = {
    "KNN":                       "knn_model.pkl",
    "Gaussian Naive Bayes":      "gnb_model.pkl",
    "SVM":                       "svm_model.pkl",
    "Random Forest":             "rf_model.pkl",
    "Artificial Neural Network": "ann_model.h5",
}

SEARCH_DIRS = ["models", "."]


def find_artifact(filename):
    for d in SEARCH_DIRS:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return candidate
    return None


def build_fallback_model(name):
    """Reasonable default estimator, used ONLY if a saved artifact for
    `name` could not be found or loaded."""
    if name == "KNN":
        return KNeighborsClassifier(n_neighbors=5)
    if name == "Gaussian Naive Bayes":
        return GaussianNB()
    if name == "SVM":
        return SVC(probability=True, random_state=RANDOM_STATE)
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    raise ValueError(f"No fallback defined for {name}")


models = {}
retrained_flags = {}

for name, filename in MODEL_FILENAMES.items():
    if name == "Artificial Neural Network":
        continue
    path = find_artifact(filename)
    if path:
        try:
            models[name] = joblib.load(path)
            retrained_flags[name] = False
            print(f"Loaded {name} from: {path}")
            continue
        except Exception as e:
            print(f"WARNING: found {name} at {path} but failed to load it ({e}).")
    print(f"Saved artifact for {name} unavailable — training a fallback model instead.")
    model = build_fallback_model(name)
    model.fit(X_train, y_train)
    models[name] = model
    retrained_flags[name] = True

# ANN is loaded/trained separately since it uses the Keras/TensorFlow format
ann_name = "Artificial Neural Network"
ann_path = find_artifact(MODEL_FILENAMES[ann_name])
try:
    from tensorflow.keras.models import load_model as load_keras_model
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense

    if ann_path:
        try:
            models[ann_name] = load_keras_model(ann_path)
            retrained_flags[ann_name] = False
            print(f"Loaded {ann_name} from: {ann_path}")
        except Exception as e:
            ann_path = None
            print(f"WARNING: found ANN artifact but failed to load it ({e}).")

    if not ann_path:
        print(f"Saved artifact for {ann_name} unavailable — training a fallback ANN instead.")
        ann = Sequential([
            Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
            Dense(16, activation="relu"),
            Dense(1, activation="sigmoid"),
        ])
        ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        ann.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
        models[ann_name] = ann
        retrained_flags[ann_name] = True
except ImportError:
    print("TensorFlow is not available in this environment — ANN cannot be loaded or trained.")
    models[ann_name] = None
    retrained_flags[ann_name] = False

print("\nModel source summary:")
for name in models:
    if models[name] is None:
        status = "unavailable"
    else:
        status = "trained as fallback (no saved artifact found)" if retrained_flags.get(name) else "loaded from saved artifact"
    print(f"  - {name}: {status}")


## 5. Evaluation Metrics

For every model, the following metrics are computed on the held-out test set:

| Metric | What it measures |
|---|---|
| **Accuracy** | Overall proportion of correct predictions |
| **Precision** | Of predicted landslides, how many were real landslides |
| **Recall** | Of actual landslides, how many were correctly detected |
| **F1-score** | Harmonic mean of Precision and Recall |
| **ROC-AUC** | Ability to separate the two classes across all thresholds |

All numerical metrics are rounded to **4 decimal places**.


In [ ]:
def get_predictions(model, X, is_ann=False):
    """Return (y_pred, y_proba) for a fitted classifier or an ANN."""
    if is_ann:
        y_proba = model.predict(X, verbose=0).ravel()
        y_pred = (y_proba >= 0.5).astype(int)
    else:
        y_pred = model.predict(X)
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X)[:, 1]
        elif hasattr(model, "decision_function"):
            y_proba = model.decision_function(X)  # e.g. SVM without probability=True
        else:
            y_proba = y_pred.astype(float)
    return y_pred, y_proba


def evaluate_model(name, model, X_test, y_test, is_ann=False):
    y_pred, y_proba = get_predictions(model, X_test, is_ann=is_ann)
    metrics = {
        "Model": name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1-score":  round(f1_score(y_test, y_pred, zero_division=0), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_proba), 4),
    }
    return metrics, y_pred, y_proba


## 6. Run Evaluation for Every Model

Each available model is scored on the identical, Phase-3-preprocessed test
set defined above, ensuring the comparison across models is fair and
consistent.


In [ ]:
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    if model is None:
        print(f"Skipping {name} — model not available.")
        continue
    is_ann = (name == "Artificial Neural Network")
    metrics, y_pred, y_proba = evaluate_model(name, model, X_test, y_test, is_ann=is_ann)
    results.append(metrics)
    predictions[name] = y_pred
    probabilities[name] = y_proba
    print(f"Evaluated {name}: F1={metrics['F1-score']:.4f}, ROC-AUC={metrics['ROC-AUC']:.4f}")

results_df = pd.DataFrame(results).set_index("Model")
results_df


## 7. Confusion Matrices

A confusion matrix is plotted for every evaluated model, showing correctly
and incorrectly classified landslide / no-landslide cases. This highlights
where each model tends to make mistakes — false negatives (missed
landslides) are the most costly error type for this use case.


In [ ]:
n_models = len(predictions)
n_cols = 3
n_rows = int(np.ceil(n_models / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["No Landslide", "Landslide"],
        yticklabels=["No Landslide", "Landslide"],
    )
    ax.set_title(name, fontsize=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

for ax in axes[len(predictions):]:
    ax.axis("off")

plt.suptitle("Confusion Matrices — All Models", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


## 8. ROC Curve Comparison

All models are plotted together on a single ROC curve so their
discriminative ability can be compared directly. The dashed diagonal line
represents a random-guess baseline (AUC = 0.5000).


In [ ]:
plt.figure(figsize=(8, 7))

for name, y_proba in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = results_df.loc[name, "ROC-AUC"]
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_score:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess (AUC = 0.5000)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison — All Models")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


## 9. Performance Comparison Table

The table below ranks all models by **F1-score** (primary), then
**ROC-AUC** (secondary), then **Accuracy** (tie-breaker). The best value in
each metric column is highlighted.


In [ ]:
comparison_df = results_df.sort_values(
    by=["F1-score", "ROC-AUC", "Accuracy"], ascending=False
).copy()

def highlight_best(col):
    is_max = col == col.max()
    return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_max]

styled_table = comparison_df.style.apply(highlight_best, subset=comparison_df.columns).format("{:.4f}")
styled_table


## 10. Best Model Selection Logic

The best model is chosen using a strict ordered rule:

1. **Primary:** highest F1-score
2. **Secondary:** highest ROC-AUC (used to break F1-score ties)
3. **Tie-breaker:** highest Accuracy (used only if F1-score and ROC-AUC are both tied)


In [ ]:
ranked_df = comparison_df.sort_values(
    by=["F1-score", "ROC-AUC", "Accuracy"], ascending=False
)

best_model_name = ranked_df.index[0]
best_row = ranked_df.loc[best_model_name]

print("Ranking (F1-score -> ROC-AUC -> Accuracy):")
print(ranked_df[["F1-score", "ROC-AUC", "Accuracy"]])
print(f"\nSelected Best Model: {best_model_name}")


## Best Model Recommendation

The cell below generates a data-driven explanation of the selection,
based directly on the metrics computed above.


In [ ]:
second_best_name = ranked_df.index[1] if len(ranked_df) > 1 else None
second_best_row = ranked_df.iloc[1] if len(ranked_df) > 1 else None

comparison_line = (
    f"For comparison, the next-best model was **{second_best_name}** "
    f"(F1-score: {second_best_row['F1-score']:.4f}, ROC-AUC: {second_best_row['ROC-AUC']:.4f}), "
    f"which trails the recommended model on the primary and/or secondary selection criteria."
    if second_best_row is not None else
    "Only one model was available for evaluation, so no direct runner-up comparison is possible."
)

recommendation = f"""
The **{best_model_name}** is recommended as the best-performing model for the
GeoSlide landslide-susceptibility classification task.

**Why {best_model_name} was selected:**

- **F1-score (primary criterion): {best_row['F1-score']:.4f}** — the highest among all
  evaluated models, reflecting the best balance between Precision
  ({best_row['Precision']:.4f}) and Recall ({best_row['Recall']:.4f}). This balance matters
  for landslide prediction: false positives create unnecessary alerts, while false negatives
  mean a real landslide risk goes undetected — F1-score penalizes models that sacrifice one
  for the other.
- **ROC-AUC (secondary criterion): {best_row['ROC-AUC']:.4f}** — confirms the model separates
  the landslide / no-landslide classes well across *all* classification thresholds, not just
  the default 0.5 cutoff, indicating robust probability estimates.
- **Accuracy (tie-breaker): {best_row['Accuracy']:.4f}** — used only to break ties when
  F1-score and ROC-AUC were equal or effectively equal between candidate models.

{comparison_line}
"""
display(Markdown(recommendation))


## Evaluation Summary

The cell below compiles the final summary: the selected best model,
overall observations across all evaluated models, and an assessment of
readiness for deployment.


In [ ]:
avg_f1 = comparison_df["F1-score"].mean()
f1_spread = comparison_df["F1-score"].max() - comparison_df["F1-score"].min()
deployment_ready = (best_row["F1-score"] >= 0.80) and (best_row["ROC-AUC"] >= 0.80)

deployment_verdict = (
    "The recommended model meets a strong performance bar "
    "(F1-score and ROC-AUC both \u2265 0.80) and is a reasonable candidate for a "
    "pilot deployment, ideally alongside human review and continuous monitoring."
    if deployment_ready else
    "The recommended model shows promising results but falls short of a strict "
    "deployment bar (F1-score and ROC-AUC \u2265 0.80). Before production deployment, "
    "consider hyperparameter tuning, addressing class imbalance, gathering more "
    "training data, or additional feature engineering."
)

summary = f"""
### Best Model
**{best_model_name}** — F1-score: {best_row['F1-score']:.4f} | ROC-AUC: {best_row['ROC-AUC']:.4f} |
Accuracy: {best_row['Accuracy']:.4f} | Precision: {best_row['Precision']:.4f} | Recall: {best_row['Recall']:.4f}

### Overall Observations
- **{len(comparison_df)} models** were evaluated on an identical, held-out test set using the
  same Phase 3 preprocessing, ensuring a fair, apples-to-apples comparison.
- The average F1-score across all models was **{avg_f1:.4f}**, with a spread of
  **{f1_spread:.4f}** between the strongest and weakest performers.
- Models where ROC-AUC is notably higher than Accuracy may benefit from threshold tuning,
  since it suggests the model separates classes well but the default 0.5 cutoff isn't optimal.
- The confusion matrices in Section 7 show where each model tends to err; minimizing false
  negatives is the priority in a landslide early-warning context, since a missed landslide is
  far costlier than a false alarm.

### Readiness for Deployment
{deployment_verdict}

**Recommended next steps before production rollout:**
1. Validate on an independent or out-of-region dataset, if available, to check geographic generalization.
2. Track the false-negative rate closely in ongoing monitoring, given the high cost of missed landslide events.
3. Schedule periodic re-evaluation as new labeled data becomes available, to catch model drift.
4. Consider threshold tuning or an ensemble approach if the Precision/Recall trade-off needs adjustment for the deployment context.
"""
display(Markdown(summary))
